# Connection to JDBC SQL Server Injection

In [0]:
server = "asql-salesproject-yash-s1.database.windows.net"
port = "1433"
database = "ASQL_SalesProject_Yash_Ingestion"
username = "sales"
password = "Yash1234"
#JDBC URL
jdbc_url_src = f"jdbc:sqlserver://{server}:{port};databaseName={database}"
#Connection Properties
connection_properties_src = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Connection to JDBC SQL Server Integration

In [0]:
server = "asql-salesproject-yash-s1.database.windows.net"
port = "1433"
database = "ASQL_SalesProject_Yash_Integration"
username = "sales"
password = "Yash1234"
#JDBC URL
jdbc_url_tgt = f"jdbc:sqlserver://{server}:{port};databaseName={database}"
#Connection Properties
connection_properties_tgt = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
%skip
#Just For reference 
# Read a Single Table
df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.watermarktable",
    properties=connection_properties
)
# Write DataFrame to SQL Server
# df.write.jdbc(
#     url=jdbc_url,
#     table="dbo.orders_output",
#     mode="overwrite",
#     properties=connection_properties
#)
df.display()

# SCD 1 Logic For all 6 tables 

In [0]:
#************** 6 tables are mention below
# 1. ORDER_METHOD src             tgt=TBL_DIM_ORDER_METHOD_LKP 
# 2.RETURN_REASON src            tgt=TBL_DIM_RETURN_REASON_LKP
# 3.COUNTRY  src                  tgt=TBL_DIM_COUNTRY_LKP
# 4.WAREHOUSE                       tgt=TBL_DIM_WAREHOUSE_LKP
# 5.Retailer                       tgt=   TBL_DIM_RETAILER_LKP
# 6.PRODUCT_NAME_LOOKUP            tgt=   TBL_DIM_PRODUCT_NAME_LKP

#reading all the tables from source and target STG
#**********************************1st.ORDER_METHOD  ************************************

df_ORDER_METHOD_src = spark.read.jdbc(url=jdbc_url_src,table="dbo.Order_Method", properties=connection_properties_src)
df_ORDER_METHOD_tgt = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_ORDER_METHOD_LKP", properties=connection_properties_tgt)
#df_stg_ORDER_METHOD = spark.read.jdbc(url=jdbc_url_tgt,table="STG.STG_ORDER_METHOD", properties=connection_properties_tgt)


#**********************************2st.RETURN_REASON  ************************************

df_RETURN_REASON_src = spark.read.jdbc(url=jdbc_url_src,table="dbo.RETURN_REASON", properties=connection_properties_src)
df_RETURN_REASON_tgt = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_RETURN_REASON_LKP", properties=connection_properties_tgt)
#df_stg_RETURN_REASON = spark.read.jdbc(url=jdbc_url_tgt,table="STG.STG_RETURN_REASON", properties=connection_properties_tgt)

#**********************************3st.COUNTRY  ************************************

df_COUNTRY_src = spark.read.jdbc(url=jdbc_url_src,table="dbo.COUNTRY", properties=connection_properties_src)
df_COUNTRY_tgt = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_COUNTRY_LKP", properties=connection_properties_tgt)
#df_stg_COUNTRY = spark.read.jdbc(url=jdbc_url_tgt,table="STG.STG_COUNTRY", properties=connection_properties_tgt)

#**********************************4st.WAREHOUSE  ************************************

df_WAREHOUSE_src = spark.read.jdbc(url=jdbc_url_src,table="dbo.WAREHOUSE", properties=connection_properties_src)
df_WAREHOUSE_tgt = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_WAREHOUSE_LKP", properties=connection_properties_tgt)
#df_stg_WAREHOUSE = spark.read.jdbc(url=jdbc_url_tgt,table="STG.STG_WAREHOUSE", properties=connection_properties_tgt)

#**********************************5st.Retailer  ************************************

df_Retailer_src = spark.read.jdbc(url=jdbc_url_src,table="dbo.RETAILER", properties=connection_properties_src)
df_Retailer_tgt = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_RETAILER_LKP", properties=connection_properties_tgt)
#df_stg_Retailer = spark.read.jdbc(url=jdbc_url_tgt,table="STG.STG_RETAILER", properties=connection_properties_tgt)

#**********************************6st.PRODUCT_NAME_LOOKUP  ************************************

df_PRODUCT_NAME_src = spark.read.jdbc(url=jdbc_url_src,table="dbo.PRODUCT_NAME", properties=connection_properties_src)
df_PRODUCT_NAME_tgt = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_PRODUCT_NAME_LKP", properties=connection_properties_tgt)
#df_stg_PRODUCT_NAME = spark.read.jdbc(url=jdbc_url_tgt,table="STG.STG_PRODUCT_NAME_LOOKUP", properties=connection_properties_tgt)

##Detect Changed Records (SCD-1)

In [0]:
from pyspark.sql.functions import *

# =========================================================
# 2. ORDER_METHOD
# BK = METHOD_CODE
# SCD1 = METHOD_NAME
# =========================================================

df_ORDER_src = df_ORDER_METHOD_src.select(col("ORDER_METHOD_CODE").alias("METHOD_CODE"),col("ORDER_METHOD_EN").alias("METHOD_NAME"),col("SOURCE_ID"),
col("DataDate").cast("date").alias("DATA_DATE"))

updated_ORDER_df = (df_ORDER_src.alias("src").join(df_ORDER_METHOD_tgt.alias("tgt"),"METHOD_CODE").filter(
        col("src.METHOD_NAME") != col("tgt.METHOD_NAME")).select("src.*"))

new_ORDER_df = (df_ORDER_src.alias("src").join(df_ORDER_METHOD_tgt.alias("tgt"),"METHOD_CODE","left_anti"))

final_ORDER_df = updated_ORDER_df.unionByName(new_ORDER_df)

# WRITE TO STG
final_ORDER_df.write.jdbc(url=jdbc_url_tgt,table="STG.STG_ORDER_METHOD",mode="overwrite",properties=connection_properties_tgt)

# =========================================================
# 3. RETURN_REASON
# BK = RETURN_REASON_CODE
# SCD1 = RETURN_REASON_DESC
# =========================================================

df_RETURN_src = df_RETURN_REASON_src.select(col("RETURN_REASON_CODE"),col("REASON_DESCRIPTION_EN").alias("RETURN_REASON_DESC"),col("SOURCE_ID"),
col("DataDate").cast("date").alias("DATA_DATE"))

updated_RETURN_df = (df_RETURN_src.alias("src").join(df_RETURN_REASON_tgt.alias("tgt"),"RETURN_REASON_CODE")
                     .filter(col("src.RETURN_REASON_DESC") != col("tgt.RETURN_REASON_DESC")).select("src.*"))

new_RETURN_df = (df_RETURN_src.alias("src").join(df_RETURN_REASON_tgt.alias("tgt"),"RETURN_REASON_CODE","left_anti"))

final_RETURN_df = updated_RETURN_df.unionByName(new_RETURN_df)

# WRITE TO STG
final_RETURN_df.write.jdbc(url=jdbc_url_tgt,table="STG.STG_RETURN_REASON",mode="overwrite",properties=connection_properties_tgt)

# =========================================================
# 4. COUNTRY
# BK = COUNTRY_CODE
# SCD1 = COUNTRY_ID, COUNTRY_NAME
# =========================================================

df_COUNTRY_src_final = df_COUNTRY_src.select(col("COUNTRY_CODE"),col("COUNTRY_ID"),col("COUNTRY_NAME"),col("SOURCE_ID"),col("DataDate").cast("date").alias("DATA_DATE"))

updated_COUNTRY_df = (df_COUNTRY_src_final.alias("src").join(df_COUNTRY_tgt.alias("tgt"),"COUNTRY_CODE")
                      .filter((col("src.COUNTRY_ID") != col("tgt.COUNTRY_ID")) |(col("src.COUNTRY_NAME") != col("tgt.COUNTRY_NAME"))).select("src.*"))

new_COUNTRY_df = (df_COUNTRY_src_final.alias("src").join(df_COUNTRY_tgt.alias("tgt"),"COUNTRY_CODE","left_anti"))

final_COUNTRY_df = updated_COUNTRY_df.unionByName(new_COUNTRY_df)

# WRITE TO STG
final_COUNTRY_df.write.jdbc(url=jdbc_url_tgt,table="STG.STG_COUNTRY",mode="overwrite",properties=connection_properties_tgt)

# =========================================================
# 5. WAREHOUSE
# BK = BRANCH_CODE
# SCD1 = ADDRESS, CITY, POSTAL_CODE, COUNTRY_CODE
# PASS THROUGH = WAREHOUSE_BRANCH_CODE
# =========================================================

df_WAREHOUSE_src_final = df_WAREHOUSE_src.select(col("BRANCH_CODE"),col("ADDRESS1").alias("ADDRESS"),col("CITY"),col("POSTAL_ZONE").alias("POSTAL_CODE"),
                                                 col("COUNTRY_CODE"),col("WAREHOUSE_BRANCH_CODE"),col("SOURCE_ID"),col("DataDate").cast("date").alias("DATA_DATE"))

updated_WAREHOUSE_df = (df_WAREHOUSE_src_final.alias("src").join(df_WAREHOUSE_tgt.alias("tgt"),"BRANCH_CODE")
                        .filter((col("src.ADDRESS") != col("tgt.ADDRESS")) |(col("src.CITY") != col("tgt.CITY")) |
                                (col("src.POSTAL_CODE") != col("tgt.POSTAL_CODE")) |(col("src.COUNTRY_CODE") != col("tgt.COUNTRY_CODE"))).select("src.*"))

new_WAREHOUSE_df = (df_WAREHOUSE_src_final.alias("src").join(df_WAREHOUSE_tgt.alias("tgt"),"BRANCH_CODE","left_anti"))

final_WAREHOUSE_df = updated_WAREHOUSE_df.unionByName(new_WAREHOUSE_df)

# WRITE TO STG
final_WAREHOUSE_df.write.jdbc(url=jdbc_url_tgt,table="STG.STG_WAREHOUSE",mode="overwrite",properties=connection_properties_tgt)

# =========================================================
# 6. RETAILER
# BK = RETAILER_SITE_CODE
# SCD1 = RETAILER_NAME, RETAILER_CONTACT_CODE
# =========================================================

df_RETAILER_src_final = df_Retailer_src.select(col("RETAILER_NAME"),col("RETAILER_SITE_CODE"),col("RETAILER_CONTACT_CODE"),col("SOURCE_ID"),
                                               col("DataDate").cast("date").alias("DATA_DATE"))

updated_RETAILER_df = (df_RETAILER_src_final.alias("src").join(df_Retailer_tgt.alias("tgt"),"RETAILER_SITE_CODE")
                       .filter((col("src.RETAILER_NAME") != col("tgt.RETAILER_NAME")) |(col("src.RETAILER_CONTACT_CODE") != col("tgt.RETAILER_CONTACT_CODE")))
                       .select("src.*"))

new_RETAILER_df = (df_RETAILER_src_final.alias("src").join(df_Retailer_tgt.alias("tgt"),"RETAILER_SITE_CODE","left_anti"))

final_RETAILER_df = updated_RETAILER_df.unionByName(new_RETAILER_df)

# WRITE TO STG
final_RETAILER_df.write.jdbc(url=jdbc_url_tgt,table="STG.STG_RETAILER",mode="overwrite",properties=connection_properties_tgt)

# =========================================================
# 7. PRODUCT_NAME
# BK = PRODUCT_NUMBER
# SCD1 = PRODUCT_NAME, PRODUCT_DESCRIPTION
# =========================================================

df_PRODUCT_src_final = df_PRODUCT_NAME_src.select(col("PRODUCT_NUMBER"),col("PRODUCT_NAME"),col("PRODUCT_DESCRIPTION"),col("SOURCE_ID"),
                                                  col("DataDate").cast("date").alias("DATA_DATE"))

updated_PRODUCT_df = (df_PRODUCT_src_final.alias("src").join(df_PRODUCT_NAME_tgt.alias("tgt"),"PRODUCT_NUMBER")
                      .filter((col("src.PRODUCT_NAME") != col("tgt.PRODUCT_NAME")) |(col("src.PRODUCT_DESCRIPTION") != col("tgt.PRODUCT_DESCRIPTION")))
                      .select("src.*"))

new_PRODUCT_df = (df_PRODUCT_src_final.alias("src").join(df_PRODUCT_NAME_tgt.alias("tgt"),"PRODUCT_NUMBER","left_anti"))

final_PRODUCT_df = updated_PRODUCT_df.unionByName(new_PRODUCT_df)

# WRITE TO STG
final_PRODUCT_df.write.jdbc(url=jdbc_url_tgt,table="STG.STG_PRODUCT_NAME_LOOKUP",mode="overwrite",properties=connection_properties_tgt)

print("All SCD1 staging loads completed successfully.")

###JOIN